# 09 · Modules, Testing & Documentation

Real programs have thousands of lines spread over many files, and they keep changing. Today you learn how to split code into **modules**, how to protect it with **tests**, and how to find answers in the **documentation**.

**By the end of this lesson you can:**
- Export and import functions between files in JavaScript and Python
- Write tests with `node:test` / `assert` and with Python's `assert`
- Make failing tests pass (test-driven development)
- Look things up on MDN and in the Python docs

In [ ]:
# ▶️ Run this cell first, every time you open the notebook.
# It lets %%node cells run JavaScript and gives you check() in both languages.
%load_ext course_helpers

## 🔁 Warm-up
Answer from memory first: **no scrolling back, no notes**. Getting it wrong is fine. Trying to remember is what makes it stick.

1. Name the three kinds of errors.
2. In which order do you read a Python traceback? A JavaScript stack trace?
3. Write a `try` / `catch` in JavaScript that catches the error from `JSON.parse("oops")` and prints its message.
4. What do the `<<<<<<<` and `>>>>>>>` lines in a file mean?

**My answers** (double-click to edit):

1. ✏️
2. ✏️
3. ✏️
4. ✏️

## Part 1: Modules
Putting everything in one file is like writing a book with no chapters. A **module** is a file of code
that other files can **import**.

You've used one all along: `%load_ext course_helpers` at the top of every notebook loads the module
`course_helpers.py` from this folder. Open it in your editor and have a look. You can read most of it now!

Let's create our own. `%%writefile` saves a cell into a file (it's a notebook feature, like `%%node`).
First, a folder for experiments:

In [ ]:
import os

os.makedirs("playground", exist_ok=True)
print("playground folder ready")

In [ ]:
%%writefile playground/mathtools.mjs
// A JavaScript module. `export` makes a function available to other files.
export function square(n) {
  return n * n;
}

export function average(numbers) {
  return numbers.reduce((sum, n) => sum + n, 0) / numbers.length;
}

// Not exported, so other files can't use it
const secret = 42;

In [ ]:
%%node
import { square, average } from "./playground/mathtools.mjs";

console.log(square(7));
console.log(average([1, 2, 3, 4]));

In [ ]:
%%writefile playground/mathtools.py
# A Python module. Everything at the top level can be imported.
def square(n):
    return n * n

def average(numbers):
    return sum(numbers) / len(numbers)

In [ ]:
from playground.mathtools import square, average

print(square(7))
print(average([1, 2, 3, 4]))

> ⚠️ Notebook quirk: Python imports a module only **once** per session. If you change `mathtools.py`,
> restart the kernel (**Restart** button) to see the change. Node reads the file fresh every time.

Languages come with many ready-made modules (the **standard library**), and millions more can be
installed with a package manager: `npm install <name>` for JavaScript and `pip install <name>` for Python.

In [ ]:
import math
import random

print(math.sqrt(16))
print(random.randint(1, 6))   # a dice roll

In [ ]:
%%node
import { platform } from "node:os";

console.log(Math.sqrt(16));                    // Math is always available
console.log(Math.floor(Math.random() * 6) + 1); // a dice roll
console.log("Running on:", platform());

## Part 2: Tests
You've used `check()` throughout the course. **Tests** are code that checks code. Why bother?
- You find bugs **before** your users do.
- You can change code later and know within a second whether you broke something.
- Tests describe what the code is *supposed* to do.

Real projects use a **test framework**. Node.js has one built in: `node:test`, with `node:assert`.

In [ ]:
%%node
import { test } from "node:test";
import assert from "node:assert/strict";
import { square, average } from "./playground/mathtools.mjs";

test("square multiplies a number by itself", () => {
  assert.equal(square(3), 9);
  assert.equal(square(-2), 4);
});

test("average of [1, 2, 3, 4] is 2.5", () => {
  assert.equal(average([1, 2, 3, 4]), 2.5);
});

test("average of an empty list is 0", () => {
  assert.equal(average([]), 0);
});

🔍 The last test **fails**: `average([])` is `NaN`, because 0 / 0 is `NaN`. The test found an edge case
we hadn't thought about. That's exactly what tests are for.

🛠️ **Fix it:** go back to the `mathtools.mjs` cell, make `average` return `0` for an empty list, run that
cell again, then run the tests again until everything is ✔.

In Python, `assert condition` does nothing when the condition is true and raises an `AssertionError`
when it's false. The most popular test tool, **pytest**, finds every function whose name starts with
`test_` and runs it. Here we call them ourselves:

In [ ]:
from playground.mathtools import square, average

def test_square():
    assert square(3) == 9
    assert square(-2) == 4

def test_average():
    assert average([1, 2, 3, 4]) == 2.5

test_square()
test_average()
print("All tests passed ✅")

## Part 3: Test first
Professionals often write the tests **first** and then the code. This is called **test-driven
development** (TDD). The tests become the specification. Your turn: the tests below are already
written. Write the module so they pass.

In [ ]:
%%writefile playground/text.mjs
// ✏️ Write the three functions so that the tests below pass.

export function capitalize(word) {
  // "hello" → "Hello"
}

export function isPalindrome(text) {
  // the same forwards and backwards, ignoring upper/lower case: "Racecar" → true
}

export function slugify(title) {
  // "Hello World" → "hello-world"  (lower case, spaces become dashes)
}

In [ ]:
%%node
// The tests. Don't change them: change the module until they pass.
import { test } from "node:test";
import assert from "node:assert/strict";
import { capitalize, isPalindrome, slugify } from "./playground/text.mjs";

test("capitalize", () => {
  assert.equal(capitalize("hello"), "Hello");
  assert.equal(capitalize("a"), "A");
});

test("isPalindrome", () => {
  assert.equal(isPalindrome("racecar"), true);
  assert.equal(isPalindrome("Racecar"), true);
  assert.equal(isPalindrome("hello"), false);
});

test("slugify", () => {
  assert.equal(slugify("Hello World"), "hello-world");
  assert.equal(slugify("Learn JavaScript Today"), "learn-javascript-today");
});

Now the same in Python. The tests are at the bottom of the cell.

In [ ]:
# ✏️ Write these functions so the tests pass

def capitalize(word):
    pass

def is_palindrome(text):
    pass

def slugify(title):
    pass


# Tests (don't change these)
def test_text_functions():
    assert capitalize("hello") == "Hello"
    assert is_palindrome("Racecar") == True
    assert is_palindrome("hello") == False
    assert slugify("Hello World") == "hello-world"

test_text_functions()
print("All tests passed ✅")

## Part 4: Reading documentation
Nobody remembers every function. Professionals look things up all the time. The best references:
- JavaScript: **MDN Web Docs**, [developer.mozilla.org](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/String)
- Python: **docs.python.org**, [String methods](https://docs.python.org/3/library/stdtypes.html#string-methods)

🛠️ Use the docs to find the method that does each thing, then use it:

In [ ]:
%%node
const filename = "report.final.pdf";
const sentence = "   too much space   ";
const word = "ha";

// ✏️ Find the right method on MDN and replace each null
const isPdf = null;       // does filename end with ".pdf"?
const trimmed = null;     // sentence without the spaces at the start and the end
const laugh = null;       // word repeated 3 times: "hahaha"
const position = null;    // at which index does ".pdf" start in filename?

check(isPdf, true);
check(trimmed, "too much space");
check(laugh, "hahaha");
check(position, 12);

In [ ]:
filename = "report.final.pdf"
sentence = "   too much space   "
word = "ha"

# ✏️ Find the right methods in the Python docs
is_pdf = None
trimmed = None
laugh = None
position = None

check(is_pdf, True)
check(trimmed, "too much space")
check(laugh, "hahaha")
check(position, 12)

## 🌍 Rosetta Stone: same idea, different language
The ideas are the same in every language. Only the spelling changes. Add your own rows when you discover something new.

| Idea | JavaScript | Python |
|---|---|---|
| Export from a module | `export function f() {}` | (everything at the top level) |
| Import from a module | `import { f } from "./file.mjs";` | `from file import f` |
| Built-in maths | `Math.sqrt(16)` | `import math` → `math.sqrt(16)` |
| Assertion | `assert.equal(a, b)` | `assert a == b` |
| Test tool | `node --test` | `pytest` |
| Install a package | `npm install name` | `pip install name` |
| Ends with / remove spaces | `s.endsWith(".pdf")` / `s.trim()` | `s.endswith(".pdf")` / `s.strip()` |
| Documentation | developer.mozilla.org | docs.python.org |

## ✍️ Explain it back
Answer in your own words, as if explaining it to a friend who missed today's session.

1. Why do we split code into modules?
2. Give two reasons to write tests.
3. What does "test-driven development" mean?

**My explanations** (double-click to edit):

1. ✏️
2. ✏️
3. ✏️

## 🌱 Git checkpoint
Run `git status`. You'll see the new `playground/` folder listed as *untracked*. Some files should
**not** go into Git:
- experiments and temporary files (like `playground/`)
- files the computer generates (`__pycache__/`, `node_modules/`)
- **secrets**: passwords and API keys, often in a file called `.env`.
  ⚠️ Anything pushed to GitHub can be copied, even if you delete it later.

The file `.gitignore` lists what Git should ignore. Open it, add a line `playground/`, save, and:

```bash
git status                # playground/ is gone from the list
git add .gitignore
git commit -m "Ignore the playground folder"
```

Then improve your `LEARNING_LOG.md` from lesson 06 with more Markdown: a heading for each lesson,
a numbered list, **bold** text, and a code example in a code block (three backticks).
Commit, push, and see how GitHub displays it.

```bash
git add LEARNING_LOG.md 09_modules_and_testing.ipynb
git commit -m "Complete lesson 09 and update learning log"
git push
```